# OpenAi Agents SDK

This library makes it straightforward to build agentic applications—where a model can use additional context and tools, hand off to other specialized agents, stream partial results, and keep a full trace of what happened.


Project: AI Customer Support System (Multi-Agent)

### Support flow

```
User
  → Router Agent
  → Planner Agent
  → ┌─────────────────────────────────────┐
    │ QnA (RAG) │ Order (APIs) │ Returns   │
    │           │              │ (workflow)│
    └─────────────────────────────────────┘
  → Human escalation (when a specialist uses the ticket tool)
```

| Agent | Role |
|-------|------|
| **Router** | First touch: off-topic or spam gets a short reply; real support requests are handed to the Planner. |
| **Planner** | Chooses **one** specialist (QnA, Order, or Returns). Does not try to answer the customer itself. |
| **QnA** | Answers from a stub “knowledge base” tool (stand-in for RAG). |
| **Order** | Stub “API” tool for order status. |
| **Returns** | Stub workflow tool; edge cases call **human escalation**. |

Under the hood this uses the SDK’s **handoffs**: control moves from agent to agent until a specialist finishes the reply.

```bash
pip install openai-agents python-dotenv
```

Set `OPENAI_API_KEY` in your environment or in a `.env` file at the project root.

In [12]:
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

load_dotenv()  # loads OPENAI_API_KEY from .env if present


# --- Stand-ins for RAG, HTTP APIs, and workflow backends ---


@function_tool
def search_knowledge_base(query: str) -> str:
    """Search help articles and policies (replace with real vector/RAG)."""
    return (
        "[KB] Returns: unworn items within 30 days. "
        "Shipping: 5–7 business days standard. "
        f"(matched: {query[:40]}…)"
    )


@function_tool
def get_order_status(order_id: str) -> str:
    """Call the orders service (replace with real API)."""
    return f"[Orders API] {order_id} — status: shipped, ETA ~2 days."


@function_tool
def start_return_workflow(order_id: str, item: str, reason: str) -> str:
    """Backend return workflow: label, refund timing, etc."""
    return (
        f"[Returns workflow] Started for {order_id} ({item}). "
        "Label email queued. Reason logged."
    )


@function_tool
def escalate_to_human(reason: str, what_happened: str) -> str:
    """Use when policy is unclear, the user wants a person, or automation cannot finish."""
    return f"[Human queue] Ticket created — {reason}. Context: {what_happened[:120]}…"


# --- Specialists (they produce the final customer-facing reply) ---

qna_agent = Agent(
    name="QnA Agent",
    handoff_description="FAQ, policies, product info — uses knowledge search (RAG).",
    instructions=(
        "Answer using search_knowledge_base when the answer is factual. "
        "If the KB does not cover it or the user needs an exception, use escalate_to_human."
    ),
    tools=[search_knowledge_base, escalate_to_human],
)

order_agent = Agent(
    name="Order Agent",
    handoff_description="Order status, tracking, order changes via backend APIs.",
    instructions=(
        "Use get_order_status for lookups. Do not invent tracking IDs. "
        "Billing disputes, chargebacks, or complex changes → escalate_to_human."
    ),
    tools=[get_order_status, escalate_to_human],
)

returns_agent = Agent(
    name="Returns Agent",
    handoff_description="Returns/refunds/exchanges: start return, wrong size, send item back, label—use even if order was discussed for other reasons.",
    instructions=(
        "For a standard return, use start_return_workflow. "
        "If damaged/high-value, policy edge case, or user insists on a person → escalate_to_human."
    ),
    tools=[start_return_workflow, escalate_to_human],
)


planner_agent = Agent(
    name="Planner Agent",
    handoff_description="Decides whether QnA, Order, or Returns should own the ticket.",
    instructions=(
        "You only route—never reply to the customer. "
        "Look at the *last* user message only for intent (earlier turns are context). "
        "Priority: (1) If it asks to return/refund/exchange, wrong size, send back, or start a return workflow "
        "→ Returns Agent, even if the same order ID was about billing earlier. "
        "(2) If it is tracking, shipping ETA, where is my order, duplicate/wrong charge, payment "
        "→ Order Agent. (3) General policy/FAQ/how long is return window "
        "→ QnA Agent. Hand off to exactly one specialist."
    ),
    handoffs=[qna_agent, order_agent, returns_agent],
)

router_agent = Agent(
    name="Router Agent",
    instructions=(
        "If the latest message is not customer support (spam, random chat, coding help), reply in one short sentence—no handoff. "
        "For shopper/support needs—including follow-ups in an existing thread—always hand off to the Planner Agent "
        "so the right specialist is chosen for the *current* request. Do not try to solve support issues yourself."
    ),
    handoffs=[planner_agent],
)


In [13]:
# Run after defining agents. Try different messages to hit QnA vs Order vs Returns.

user_message = "Where is order ORD-4421?"  # → likely Order Agent
# user_message = "Start a return for ORD-9912, blue boots, too small"  # → Returns Agent
# user_message = "What's your return window if I only wore the shoes indoors?"  # → QnA Agent

result = await Runner.run(router_agent, user_message)
print(result.final_output)
print("→ Answered by:", result.last_agent.name)


Order ORD-4421 has been shipped and is expected to arrive in about 2 days. If you need more details or tracking information, let me know!
→ Answered by: Order Agent


### Multi-turn chat & human escalation

**History:** append each new user line to **`result.to_input_list()`** so the model sees the full thread.

**Two ways to continue (toggle in the next cell):**

| Mode | `Runner.run(...)` | When to use |
|------|-------------------|-------------|
| **Re-route (production-style)** | **`router_agent`** + full history | Router → Planner run again so the *latest* line *can* go to a new specialist. **Not a guarantee:** the model may still pick the same agent (e.g. Order twice). Strong prompts, or a small classifier, improve consistency. |
| **Sticky specialist** | **`result.last_agent`** + history | Same topic, same owner (e.g. “What’s tracking?” → “When exactly?”). Cheaper, no re-triage. |

**`input()`** in notebooks often breaks in Cursor/VS Code (no stdin). Edit **`follow_up_messages`** and re-run the cell instead.

In [14]:
# Edit follow_up_messages, pick a mode, run once.
INTERACTIVE = True  # False = single scripted second turn only (still uses RE_ROUTE_EACH_TURN)

# True → each new user line goes through Router → Planner again (realistic for mixed topics).
# False → continue with whoever answered last (good for same-topic follow-ups only).
RE_ROUTE_EACH_TURN = True

follow_up_messages = [
    "Thanks — can a human call me about the double charge?",
    "Also start a return for the sneakers on ORD-5560, wrong size.",
]

opening = (
    "Order ORD-5560 — I was charged twice. I want a real person to fix this, not a bot."
)

result = await Runner.run(router_agent, opening)
print("--- Turn 1 ---")
print(result.final_output)
print("→ Agent:", result.last_agent.name, "| re-route mode:", RE_ROUTE_EACH_TURN, "\n")

if INTERACTIVE:
    for i, line in enumerate(follow_up_messages, start=2):
        line = line.strip()
        if not line:
            continue
        history = result.to_input_list()
        history.append({"role": "user", "content": line})
        nxt = router_agent if RE_ROUTE_EACH_TURN else result.last_agent
        result = await Runner.run(nxt, history)
        print(f"--- Turn {i} ---")
        print("User:", line)
        print(result.final_output)
        print("→ Agent:", result.last_agent.name, "\n")
else:
    follow_up = "Also start a return for the sneakers on ORD-5560 — wrong size."
    history = result.to_input_list()
    history.append({"role": "user", "content": follow_up})
    nxt = router_agent if RE_ROUTE_EACH_TURN else result.last_agent
    result = await Runner.run(nxt, history)
    print("--- Turn 2 (scripted) ---")
    print(result.final_output)
    print("→ Agent:", result.last_agent.name)

--- Turn 1 ---
I've created a support ticket and escalated your issue to a real person, as you requested. A customer service representative will review your order (ORD-5560) and contact you directly to resolve the double charge. You won't have to deal with a bot—someone will help you as soon as possible.
→ Agent: Order Agent | re-route mode: True 

--- Turn 2 ---
User: Thanks — can a human call me about the double charge?
Your request for a call from a human representative regarding the double charge has been noted. I’ve escalated your issue and included your preference to be contacted by phone. A real person from customer support will reach out to you as soon as possible to resolve this. If you’d like to provide a preferred phone number or time for the call, please let me know!
→ Agent: Order Agent 

--- Turn 3 ---
User: Also start a return for the sneakers on ORD-5560, wrong size.
I've started a return for the sneakers from order ORD-5560 due to the wrong size. You should receive a r

### Request flow (this notebook)

**Re-route mode (`RE_ROUTE_EACH_TURN = True`):** every new user line starts again at **Router** → **Planner** → one specialist (full **history** is still passed in).

**Sticky mode (`False`):** the next turn starts at **`last_agent`** only—no new Router/Planner pass.

```mermaid
flowchart TD
  subgraph one_turn ["Each new user message"]
    H["History = to_input_list() + new user line"]
    D{"RE_ROUTE_EACH_TURN?"}
    H --> D
  end

  D -->|true| R["Router Agent"]
  D -->|false| L["last_agent"]

  R --> RQ{"Support / shopper?"}
  RQ -->|no| S["Short reply — done"]
  RQ -->|yes| P["Planner Agent"]

  P --> Q["QnA Agent\n(RAG tools)"]
  P --> O["Order Agent\n(API tools)"]
  P --> RT["Returns Agent\n(workflow tools)"]

  L --> A["Same specialist\n+ tools"]

  Q --> E["Optional:\nescalate_to_human"]
  O --> E
  RT --> E
  A --> E

  Q --> OUT["User sees final_output"]
  O --> OUT
  RT --> OUT
  A --> OUT
```

ASCII (same idea):

```
                    ┌─────────────────────────────┐
  new user line ──► │ history = prior + new line   │
                    └─────────────┬───────────────┘
                                  │
                    ┌─────────────▼──────────────┐
                    │  RE_ROUTE_EACH_TURN ?     │
                    └───┬─────────────────┬─────┘
                  yes   │                 │ no
                        ▼                 ▼
                 Router Agent      last_agent
                        │          (sticky owner)
            support? ───┼─── no ──► short reply
                        │ yes
                        ▼
                 Planner Agent
                        │
         ┌──────────────┼──────────────┐
         ▼              ▼              ▼
    QnA (RAG)    Order (APIs)   Returns (workflow)
         │              │              │
         └──────────────┴──────────────┴──► reply (+ optional human ticket)
```